# NeuroRelAD — Neuro-Motor Relational and Adaptive Development Model
## End-to-end research implementation notebook

This notebook provides an auditable implementation scaffold for the Infant Motor–Brain Multimodal Database (IMB-MD) and the NeuroRelAD pipeline: data harmonization, paired-modality checks, movement/MRI feature extraction, neuro-motor fusion, calibrated CP-risk estimation, evidence-grounded pathway ranking, longitudinal updating, model optimization, and subject-level validation.

**Important data-integrity rules**
- The notebook does not fabricate real participants, MRI scans, movement recordings, clinical labels, or Kaggle uploads.
- Supply a legitimately collected, consented/de-identified dataset in which the movement sequence and MRI are verified to belong to the same child. The notebook uses IMB-MD as the sole database name throughout.
- The manifest's `linkage_status` must be set by the data custodian after record-level verification; matching IDs alone do not prove that two modalities belong to the same child.
- This is research software, not a diagnostic device. CP-risk estimates and pathway rankings require external clinical validation and clinician review. Do not use model-generated recommendations as treatment instructions.
- For valid evaluation, split by participant, not by frame, scan, or visit. Do not use synthetic/demo data to report clinical performance.

## 1. Runtime and configuration

Designed for Google Colab or a Python environment. Set `DATA_ROOT` to the directory containing the IMB-MD manifest and modality files. Expected manifest: `participants.csv` (one row per assessment/session), with the columns documented below. MRI files can be NIfTI (`.nii`, `.nii.gz`); motion can be `.npy`, `.npz`, or CSV pose tables.

Recommended folder layout (BIDS-inspired, not claiming formal BIDS compliance):
```text
IMB-MD/
  participants.csv
  sub-0001/ses-01/anat/T1w.nii.gz
  sub-0001/ses-01/motion/pose3d.npy
  sub-0001/ses-01/derivatives/lesion_mask.nii.gz
```

In [ ]:
# Uncomment in Colab if needed:
# %pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib nibabel

from pathlib import Path
import os, json, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_ROOT = Path("/content/IMB-MD")  # change to your mounted dataset directory
MANIFEST_PATH = DATA_ROOT / "participants.csv"
OUTPUT_DIR = DATA_ROOT / "neurorelad_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Data root:", DATA_ROOT)
print("Manifest exists:", MANIFEST_PATH.exists())

## 2. IMB-MD manifest contract

Required columns:
- `participant_id`: stable de-identified child identifier
- `session_id`: visit identifier
- `assessment_date`: ISO date, if available
- `age_months`: age at assessment
- `cp_label`: 1 = CP label, 0 = no CP label; blank when unavailable
- `motion_path`: relative or absolute path to 3D pose sequence
- `mri_path`: relative or absolute path to MRI volume
- `lesion_mask_path`: optional path to lesion mask
- `linkage_status`: `verified`, `unverified`, or `not_paired`
- `label_source`: provenance for the clinical label
- `label_version`: version/date of label definition or annotation protocol

Optional fields can include `site_id`, `sex_at_birth` (only if approved and justified), `scanner`, `sequence`, `annotator_id`, `annotation_date`, and `notes`. Avoid unnecessary direct identifiers. Keep the re-identification key outside this dataset.

In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Place your real IMB-MD manifest at {MANIFEST_PATH}. "
        "Do not use generated/demo rows for clinical performance claims."
    )

df = pd.read_csv(MANIFEST_PATH, dtype={"participant_id": "string", "session_id": "string"})
required = [
    "participant_id", "session_id", "age_months", "cp_label",
    "motion_path", "mri_path", "linkage_status", "label_source"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Manifest is missing required columns: {missing}")

for c in ["assessment_date", "label_version", "lesion_mask_path"]:
    if c not in df.columns:
        df[c] = pd.NA

def resolve_path(value):
    if pd.isna(value) or str(value).strip() == "":
        return None
    p = Path(str(value))
    return p if p.is_absolute() else DATA_ROOT / p

for c in ["motion_path", "mri_path", "lesion_mask_path"]:
    df[c + "_resolved"] = df[c].map(resolve_path)

df["age_months"] = pd.to_numeric(df["age_months"], errors="coerce")
df["cp_label"] = pd.to_numeric(df["cp_label"], errors="coerce")
df["assessment_date"] = pd.to_datetime(df["assessment_date"], errors="coerce")
df["linkage_status"] = df["linkage_status"].astype("string").str.lower().str.strip()

print("Rows:", len(df), "| Unique participants:", df["participant_id"].nunique())
display(df.head())

## 3. Stage 1 — subject-level harmonization, linkage audit, provenance, and modality availability

In [ ]:
# Audit missing IDs, duplicate participant/session rows, labels, files, and declared linkage status.
df["duplicate_session"] = df.duplicated(["participant_id", "session_id"], keep=False)
df["motion_exists"] = df["motion_path_resolved"].map(lambda p: bool(p and p.exists()))
df["mri_exists"] = df["mri_path_resolved"].map(lambda p: bool(p and p.exists()))
df["lesion_mask_exists"] = df["lesion_mask_path_resolved"].map(lambda p: bool(p and p.exists()))

df["modality_status"] = np.select(
    [df["motion_exists"] & df["mri_exists"],
     df["motion_exists"] & ~df["mri_exists"],
     ~df["motion_exists"] & df["mri_exists"]],
    ["movement_and_MRI", "movement_only", "MRI_only"],
    default="neither_available"
)
df["paired_verified"] = (
    df["modality_status"].eq("movement_and_MRI")
    & df["linkage_status"].eq("verified")
    & df["participant_id"].notna()
    & df["session_id"].notna()
)

# Labels outside {0,1} are invalid for this binary endpoint; retain missing labels for unsupervised/longitudinal use.
bad_label = df["cp_label"].notna() & ~df["cp_label"].isin([0, 1])
if bad_label.any():
    warnings.warn(f"{int(bad_label.sum())} rows have cp_label outside 0/1; resetting those labels to missing.")
    df.loc[bad_label, "cp_label"] = np.nan

audit = {
    "n_rows": int(len(df)),
    "n_participants": int(df["participant_id"].nunique()),
    "duplicate_participant_session_rows": int(df["duplicate_session"].sum()),
    "missing_participant_id": int(df["participant_id"].isna().sum()),
    "missing_age_months": int(df["age_months"].isna().sum()),
    "movement_and_MRI_files_present": int(df["modality_status"].eq("movement_and_MRI").sum()),
    "verified_paired_rows": int(df["paired_verified"].sum()),
    "movement_only_rows": int(df["modality_status"].eq("movement_only").sum()),
    "MRI_only_rows": int(df["modality_status"].eq("MRI_only").sum()),
    "neither_available_rows": int(df["modality_status"].eq("neither_available").sum()),
    "missing_label_source": int(df["label_source"].isna().sum()),
}
(OUTPUT_DIR / "stage1_audit.json").write_text(json.dumps(audit, indent=2))
display(pd.Series(audit, name="count").to_frame())
display(df["modality_status"].value_counts().rename_axis("status").to_frame("rows"))

In [ ]:
# Keep one row per participant/session only after the duplicates have been reviewed.
# Do not silently choose among conflicting duplicate records.
if df["duplicate_session"].any():
    duplicate_rows = df.loc[df["duplicate_session"]].sort_values(["participant_id", "session_id"])
    duplicate_rows.to_csv(OUTPUT_DIR / "duplicate_sessions_for_review.csv", index=False)
    print("Duplicate sessions exported for review:", OUTPUT_DIR / "duplicate_sessions_for_review.csv")

# A provenance ledger is preserved for every label and optional lesion annotation.
provenance_cols = [
    c for c in ["participant_id", "session_id", "assessment_date", "cp_label",
                "label_source", "label_version", "lesion_mask_path", "linkage_status"]
    if c in df.columns
]
df[provenance_cols].to_csv(OUTPUT_DIR / "annotation_provenance.csv", index=False)
df.to_csv(OUTPUT_DIR / "harmonized_manifest.csv", index=False)

## 4. Load and summarize 3D movement sequences

In [ ]:
def load_motion(path):
    """Load a pose sequence as (T, J, C) or a 2D table. C usually contains x,y,z.
    Expected .npy shape: (frames, joints, 3) or (frames, features).
    .npz: first array is used unless a key named 'poses' exists.
    CSV: numeric columns are treated as flattened pose features.
    """
    if path is None or not Path(path).exists():
        return None
    path = Path(path)
    if path.suffix.lower() == ".npy":
        arr = np.load(path, allow_pickle=False)
    elif path.suffix.lower() == ".npz":
        z = np.load(path, allow_pickle=False)
        key = "poses" if "poses" in z.files else z.files[0]
        arr = z[key]
    elif path.suffix.lower() == ".csv":
        arr = pd.read_csv(path).select_dtypes(include=[np.number]).to_numpy()
    else:
        raise ValueError(f"Unsupported motion file: {path}")
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim not in (2, 3):
        raise ValueError(f"Motion array must be 2D or 3D, got {arr.shape} in {path}")
    if arr.shape[0] < 2:
        raise ValueError(f"Motion sequence needs at least 2 frames: {path}")
    return arr

def motion_features(arr, prefix="mot"):
    """Compact robust temporal descriptors; preserves missingness as explicit features."""
    x = np.asarray(arr, dtype=np.float32)
    if x.ndim == 3:
        # Flatten joints/coordinates per frame; retain coordinate-wise statistics.
        x = x.reshape(x.shape[0], -1)
    finite = np.isfinite(x)
    missing_fraction = 1.0 - float(finite.mean())
    x = np.where(finite, x, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        mean = np.nanmean(x, axis=0)
        std = np.nanstd(x, axis=0)
        dif = np.diff(x, axis=0)
        speed_mean = np.nanmean(np.abs(dif), axis=0)
        speed_std = np.nanstd(dif, axis=0)
    def safe(v):
        return np.nan_to_num(np.asarray(v), nan=0.0, posinf=0.0, neginf=0.0)
    feats = {
        f"{prefix}_missing_fraction": missing_fraction,
        f"{prefix}_n_frames": float(x.shape[0]),
        f"{prefix}_n_features": float(x.shape[1]),
    }
    # Cap feature dimensionality to avoid thousands of columns from large skeletons.
    for name, vec in [("median", med), ("iqr", q75-q25), ("mean", mean),
                      ("std", std), ("abs_velocity_mean", speed_mean),
                      ("velocity_std", speed_std)]:
        for i, val in enumerate(safe(vec)[:256]):
            feats[f"{prefix}_{name}_{i:03d}"] = float(val)
    return feats

motion_rows = []
for idx, row in df.iterrows():
    path = row["motion_path_resolved"]
    try:
        arr = load_motion(path)
        feats = motion_features(arr) if arr is not None else {}
        feats.update({"_row_index": idx, "participant_id": row["participant_id"],
                      "session_id": row["session_id"]})
        motion_rows.append(feats)
    except Exception as e:
        motion_rows.append({"_row_index": idx, "participant_id": row["participant_id"],
                            "session_id": row["session_id"], "motion_error": str(e)})

motion_df = pd.DataFrame(motion_rows)
motion_df.to_csv(OUTPUT_DIR / "movement_features.csv", index=False)
print("Movement feature table:", motion_df.shape)
display(motion_df.head())

## 5. MRI and lesion-mask feature extraction

In [ ]:
def load_nifti(path):
    try:
        import nibabel as nib
    except ImportError as e:
        raise ImportError("Install nibabel first: %pip install -q nibabel") from e
    img = nib.load(str(path))
    vol = np.asarray(img.get_fdata(dtype=np.float32))
    return vol, img

def volume_features(vol, prefix="mri"):
    """Basic intensity/volume descriptors, not a substitute for validated segmentation or radiomics."""
    x = np.asarray(vol, dtype=np.float32)
    finite = np.isfinite(x)
    vals = x[finite]
    if vals.size == 0:
        raise ValueError("MRI volume contains no finite voxels")
    nz = vals[vals != 0]
    use = nz if nz.size > 100 else vals
    feats = {
        f"{prefix}_dim_x": float(x.shape[0]),
        f"{prefix}_dim_y": float(x.shape[1]),
        f"{prefix}_dim_z": float(x.shape[2]) if x.ndim > 2 else 1.0,
        f"{prefix}_finite_fraction": float(finite.mean()),
        f"{prefix}_nonzero_fraction": float(np.mean(x != 0)),
    }
    for q in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
        feats[f"{prefix}_intensity_p{q:02d}"] = float(np.percentile(use, q))
    feats[f"{prefix}_intensity_mean"] = float(np.mean(use))
    feats[f"{prefix}_intensity_std"] = float(np.std(use))
    feats[f"{prefix}_intensity_skew_proxy"] = float(
        np.mean(((use - np.mean(use)) / (np.std(use) + 1e-8)) ** 3)
    )
    return feats

def mask_features(mask, prefix="lesion"):
    m = np.asarray(mask)
    binary = np.isfinite(m) & (m > 0)
    return {
        f"{prefix}_mask_available": 1.0,
        f"{prefix}_voxel_fraction": float(binary.mean()),
        f"{prefix}_voxel_count": float(binary.sum()),
    }

mri_rows = []
for idx, row in df.iterrows():
    feats = {"_row_index": idx, "participant_id": row["participant_id"],
             "session_id": row["session_id"], "mri_feature_available": 0.0}
    try:
        path = row["mri_path_resolved"]
        if path is not None and path.exists():
            vol, img = load_nifti(path)
            feats.update(volume_features(vol))
            feats["mri_feature_available"] = 1.0
            feats["mri_voxel_spacing_x"] = float(img.header.get_zooms()[0])
            feats["mri_voxel_spacing_y"] = float(img.header.get_zooms()[1])
            feats["mri_voxel_spacing_z"] = float(img.header.get_zooms()[2]) if len(img.header.get_zooms()) > 2 else 1.0
            mask_path = row.get("lesion_mask_path_resolved")
            if mask_path is not None and Path(mask_path).exists():
                mask, _ = load_nifti(mask_path)
                if mask.shape != vol.shape:
                    feats["lesion_mask_shape_match"] = 0.0
                else:
                    feats.update(mask_features(mask))
                    feats["lesion_mask_shape_match"] = 1.0
    except Exception as e:
        feats["mri_error"] = str(e)
    mri_rows.append(feats)

mri_df = pd.DataFrame(mri_rows)
mri_df.to_csv(OUTPUT_DIR / "mri_features.csv", index=False)
print("MRI feature table:", mri_df.shape)
display(mri_df.head())

**Interpretation note:** the included MRI descriptors are transparent baseline features, not anatomical region segmentation. For region-level claims, provide validated atlas/segmentation masks in the same image space, document registration and QC, then add per-region features. The code below supports lesion-mask summaries only when a mask is available and shape-aligned.

## 6. Assemble the session-level multimodal table

In [ ]:
key = ["_row_index", "participant_id", "session_id"]
meta = df.reset_index(names="_row_index")
features = meta.merge(motion_df, on=key, how="left", suffixes=("", "_motion"))
features = features.merge(mri_df, on=key, how="left", suffixes=("", "_mri"))

# Keep only rows whose modalities are explicitly verified and whose files/features are available
features["paired_for_fusion"] = (
    features["paired_verified"].fillna(False)
    & features["mri_feature_available"].fillna(0).eq(1)
    & features.filter(regex=r"^mot_").notna().any(axis=1)
)
features.to_csv(OUTPUT_DIR / "session_level_multimodal_features.csv", index=False)
print("Verified feature-ready pairs:", int(features["paired_for_fusion"].sum()))
display(features[["participant_id", "session_id", "age_months", "cp_label",
                  "paired_verified", "paired_for_fusion"]].head())

## 7. Stage 2 — stable neuro-motor relationships (sCCA + stability selection)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils import resample

motion_cols = [c for c in features.columns if c.startswith("mot_") and pd.api.types.is_numeric_dtype(features[c])]
mri_cols = [c for c in features.columns if c.startswith("mri_") and pd.api.types.is_numeric_dtype(features[c])]
paired = features.loc[features["paired_for_fusion"]].copy()
paired = paired.loc[paired["cp_label"].isin([0, 1])].copy()

if len(paired) < 10 or paired["cp_label"].nunique() < 2:
    raise ValueError("Need at least 10 verified paired, labeled sessions and both CP-label classes for this baseline. Collect more valid data; do not synthesize participants.")

# Remove constant/all-missing columns before fitting.
def usable_columns(frame, cols):
    return [c for c in cols if frame[c].notna().any() and frame[c].nunique(dropna=True) > 1]

motion_cols = usable_columns(paired, motion_cols)
mri_cols = usable_columns(paired, mri_cols)
if not motion_cols or not mri_cols:
    raise ValueError("No usable features in one or both modalities.")

# Subject-level split is used for all model evaluation. Fit all preprocessing on train only.
from sklearn.model_selection import GroupShuffleSplit
groups = paired["participant_id"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(paired, paired["cp_label"], groups))
train = paired.iloc[train_idx].copy()
test = paired.iloc[test_idx].copy()

# For sCCA-style sparse relationship discovery, use L1-regularized projections as a practical baseline.
# Select alpha/regularization only inside training data in a future nested-CV study.
def scaled_impute_fit(train_frame, test_frame, cols):
    imp = SimpleImputer(strategy="median", add_indicator=True)
    scaler = StandardScaler()
    Xtr = imp.fit_transform(train_frame[cols])
    Xte = imp.transform(test_frame[cols])
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)
    return Xtr, Xte, imp, scaler

Xm_tr, Xm_te, motion_imputer, motion_scaler = scaled_impute_fit(train, test, motion_cols)
Xr_tr, Xr_te, mri_imputer, mri_scaler = scaled_impute_fit(train, test, mri_cols)

n_comp = max(1, min(3, Xm_tr.shape[1], Xr_tr.shape[1], len(train)-1))
cca = CCA(n_components=n_comp, max_iter=1000, scale=False)
U_tr, V_tr = cca.fit_transform(Xm_tr, Xr_tr)
U_te, V_te = cca.transform(Xm_te, Xr_te)
corrs = [float(np.corrcoef(U_tr[:, i], V_tr[:, i])[0, 1]) for i in range(n_comp)]
pd.DataFrame({"component": np.arange(1, n_comp+1), "train_canonical_correlation": corrs}).to_csv(
    OUTPUT_DIR / "scca_relationships.csv", index=False
)
print("Training canonical correlations:", np.round(corrs, 4))

## 8. Stage 3 — NMR-Fuse: low-rank bilinear fusion, group regularization proxy, gated residual, calibrated risk

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, brier_score_loss,
                             confusion_matrix, log_loss)
from sklearn.base import clone

def low_rank_bilinear(A, B, rank=8):
    """Compact elementwise bilinear interaction via low-rank projections."""
    rank = max(1, min(rank, A.shape[1], B.shape[1]))
    # Fixed reproducible projections; supervised fitting happens in the classifier.
    rng = np.random.default_rng(RANDOM_STATE)
    PA = rng.normal(0, 1/np.sqrt(A.shape[1]), (A.shape[1], rank))
    PB = rng.normal(0, 1/np.sqrt(B.shape[1]), (B.shape[1], rank))
    return (A @ PA) * (B @ PB)

def fused_features(A, B):
    interaction = low_rank_bilinear(A, B, rank=min(16, A.shape[1], B.shape[1]))
    # Gated residual proxy: bounded gates estimated from modality summary magnitudes.
    ga = 1.0 / (1.0 + np.exp(-np.clip(np.mean(np.abs(A), axis=1, keepdims=True), -20, 20)))
    gb = 1.0 / (1.0 + np.exp(-np.clip(np.mean(np.abs(B), axis=1, keepdims=True), -20, 20)))
    return np.hstack([ga*A, gb*B, interaction, U_tr[:len(A)] if len(A)==len(U_tr) else np.zeros((len(A), n_comp))])

# Build train/test interaction features independently with the same deterministic projection.
# Canonical scores are included from the fitted training-only CCA transform.
def make_fused(A, B, U):
    inter = low_rank_bilinear(A, B, rank=min(16, A.shape[1], B.shape[1]))
    ga = 1 / (1 + np.exp(-np.clip(np.mean(np.abs(A), axis=1, keepdims=True), -20, 20)))
    gb = 1 / (1 + np.exp(-np.clip(np.mean(np.abs(B), axis=1, keepdims=True), -20, 20)))
    return np.hstack([ga*A, gb*B, inter, U])

F_tr = make_fused(Xm_tr, Xr_tr, U_tr)
F_te = make_fused(Xm_te, Xr_te, U_te)
y_tr = train["cp_label"].astype(int).to_numpy()
y_te = test["cp_label"].astype(int).to_numpy()

# L1 logistic regression is a sparse/group-redundancy suppression baseline.
base = LogisticRegression(penalty="l1", solver="liblinear", C=0.1, class_weight="balanced",
                          max_iter=3000, random_state=RANDOM_STATE)
base.fit(F_tr, y_tr)
calibrator = CalibratedClassifierCV(base, method="sigmoid", cv=3)
calibrator.fit(F_tr, y_tr)
risk = calibrator.predict_proba(F_te)[:, 1]
pred = (risk >= 0.5).astype(int)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if tn + fp else np.nan

metrics = {
    "n_train_sessions": int(len(train)), "n_test_sessions": int(len(test)),
    "n_train_participants": int(train["participant_id"].nunique()),
    "n_test_participants": int(test["participant_id"].nunique()),
    "accuracy": float(accuracy_score(y_te, pred)),
    "precision": float(precision_score(y_te, pred, zero_division=0)),
    "recall_sensitivity": float(recall_score(y_te, pred, zero_division=0)),
    "specificity": float(specificity_score(y_te, pred)),
    "f1": float(f1_score(y_te, pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_te, risk)) if len(np.unique(y_te)) == 2 else np.nan,
    "average_precision": float(average_precision_score(y_te, risk)) if len(np.unique(y_te)) == 2 else np.nan,
    "brier_score": float(brier_score_loss(y_te, risk)),
    "log_loss": float(log_loss(y_te, np.c_[1-risk, risk], labels=[0,1])),
}
(OUTPUT_DIR / "nmr_fuse_metrics.json").write_text(json.dumps(metrics, indent=2))
display(pd.Series(metrics, name="value").to_frame())
pd.DataFrame({"participant_id": test["participant_id"].to_numpy(),
              "session_id": test["session_id"].to_numpy(),
              "cp_label": y_te, "estimated_risk": risk, "predicted_label": pred}).to_csv(
    OUTPUT_DIR / "heldout_predictions.csv", index=False
)

## 9. Contribution scores and selected cross-modal associations

In [ ]:
# Approximate modality contribution by permutation on held-out participants.
# This measures predictive reliance for this fitted model, not causality.
from sklearn.metrics import roc_auc_score

def permutation_auc_drop(A, B, U, y, fitted_model, repeats=10):
    baseline_X = make_fused(A, B, U)
    baseline = roc_auc_score(y, fitted_model.predict_proba(baseline_X)[:, 1])
    rng = np.random.default_rng(RANDOM_STATE)
    drops = {"movement": [], "mri": []}
    for _ in range(repeats):
        p = rng.permutation(len(y))
        for name, Ap, Bp in [("movement", A[p], B), ("mri", A, B[p])]:
            Xp = make_fused(Ap, Bp, U)
            drops[name].append(baseline - roc_auc_score(y, fitted_model.predict_proba(Xp)[:, 1]))
    return {k: {"mean_auc_drop": float(np.mean(v)), "std_auc_drop": float(np.std(v))}
            for k, v in drops.items()}

try:
    contribution = permutation_auc_drop(Xm_te, Xr_te, U_te, y_te, calibrator)
except Exception as e:
    contribution = {"note": f"Contribution analysis unavailable: {e}"}
(OUTPUT_DIR / "modality_contribution.json").write_text(json.dumps(contribution, indent=2))
print(json.dumps(contribution, indent=2))

# Report canonical correlations as associations only. Do not interpret them as causal brain-movement effects.

## 10. Stage 4 — evidence-based clinical pathway knowledge base

In [ ]:
# Populate this file from applicable, authoritative clinical guidance and local care pathways.
# Entries below are schema examples only and intentionally not populated with clinical advice.
PATHWAY_FILE = DATA_ROOT / "clinical_pathways.json"
pathway_schema = {
  "version": "replace-with-reviewed-version",
  "review_date": "YYYY-MM-DD",
  "reviewers": ["clinician-role-or-team"],
  "options": [
    {
      "option_id": "unique-id",
      "name": "clinician-approved assessment or monitoring option",
      "min_age_months": 0,
      "max_age_months": 999,
      "indications": ["documented indication"],
      "contraindications": [],
      "evidence_strength": 0.0,
      "clinical_relevance": 0.0,
      "feasibility": 0.0,
      "source_title": "authoritative guideline title",
      "source_url": "https://...",
      "source_version_or_date": "version/date",
      "applicability_notes": "scope, population, limitations"
    }
  ]
}
if not PATHWAY_FILE.exists():
    PATHWAY_FILE.write_text(json.dumps(pathway_schema, indent=2))
    print("Created pathway schema. A qualified clinical team must populate and review it:", PATHWAY_FILE)
else:
    print("Pathway file found:", PATHWAY_FILE)

pathway_db = json.loads(PATHWAY_FILE.read_text())

## 11. Stage 5 — DPRE: context-aware candidate filtering, entropy weights, constrained TOPSIS

In [ ]:
def entropy_weights(matrix):
    """Entropy criterion weights for nonnegative decision matrix (rows=options, cols=criteria)."""
    X = np.asarray(matrix, dtype=float)
    X = np.clip(X, 0, None)
    col_sum = X.sum(axis=0, keepdims=True)
    P = np.divide(X, col_sum, out=np.zeros_like(X), where=col_sum > 0)
    n = X.shape[0]
    if n <= 1:
        return np.ones(X.shape[1]) / X.shape[1]
    k = 1.0 / np.log(n)
    entropy = -k * np.sum(np.where(P > 0, P*np.log(np.clip(P, 1e-12, 1)), 0), axis=0)
    divergence = 1 - entropy
    return divergence / divergence.sum() if divergence.sum() > 0 else np.ones(X.shape[1])/X.shape[1]

def topsis_rank(options, criteria, weights):
    X = options[criteria].astype(float).to_numpy()
    norm = np.linalg.norm(X, axis=0)
    R = X / np.where(norm == 0, 1, norm)
    V = R * weights
    ideal_best, ideal_worst = V.max(axis=0), V.min(axis=0)
    d_best = np.linalg.norm(V - ideal_best, axis=1)
    d_worst = np.linalg.norm(V - ideal_worst, axis=1)
    score = d_worst / np.where(d_best + d_worst == 0, 1, d_best + d_worst)
    result = options.copy()
    result["topsis_score"] = score
    return result.sort_values("topsis_score", ascending=False)

def recommend_pathways(age_months, risk_score, documented_context=None, uncertainty=0.0):
    """Ranks reviewed pathway options; must be reviewed by a clinician before use."""
    documented_context = documented_context or {}
    options = pd.DataFrame(pathway_db.get("options", []))
    if options.empty:
        return pd.DataFrame(), "No clinician-reviewed pathway options are configured."
    required = ["option_id", "name", "min_age_months", "max_age_months",
                "evidence_strength", "clinical_relevance", "feasibility",
                "source_title", "source_url"]
    missing = [c for c in required if c not in options]
    if missing:
        return pd.DataFrame(), f"Pathway knowledge base missing fields: {missing}"
    eligible = options[
        (pd.to_numeric(options["min_age_months"], errors="coerce") <= age_months)
        & (pd.to_numeric(options["max_age_months"], errors="coerce") >= age_months)
    ].copy()
    # Apply explicit contraindications/context constraints; no model inference can override these.
    def is_eligible(row):
        contraindications = row.get("contraindications", [])
        if not isinstance(contraindications, list):
            contraindications = []
        documented_exclusions = set(documented_context.get("contraindications", []))
        return not bool(documented_exclusions.intersection(set(map(str, contraindications))))
    eligible = eligible[eligible.apply(is_eligible, axis=1)]
    if eligible.empty:
        return eligible, "No eligible options after age/context constraints; clinician review required."
    criteria = ["clinical_relevance", "evidence_strength", "feasibility"]
    for c in criteria:
        eligible[c] = pd.to_numeric(eligible[c], errors="coerce").fillna(0).clip(0, 1)
    weights = entropy_weights(eligible[criteria].to_numpy())
    ranked = topsis_rank(eligible, criteria, weights)
    # Conservative uncertainty penalty; not a validated clinical decision threshold.
    ranked["uncertainty"] = float(np.clip(uncertainty, 0, 1))
    ranked["uncertainty_adjusted_score"] = ranked["topsis_score"] * (1 - ranked["uncertainty"])
    ranked = ranked.sort_values("uncertainty_adjusted_score", ascending=False)
    ranked["review_flag"] = "Clinician review required"
    ranked["risk_context_for_review_only"] = float(risk_score)
    return ranked, "Ranking is decision support only; verify source applicability and review clinically."

# Example usage after a real prediction:
# ranked, note = recommend_pathways(age_months=8, risk_score=float(risk[0]), uncertainty=0.25)
# display(ranked); print(note)

## 12. Stage 6–7 — longitudinal state update, change-point signal, uncertainty, and recommendation stability

In [ ]:
def longitudinal_update(previous_risk, new_risk, previous_variance, observation_reliability=1.0):
    """Simple scalar Bayesian-style update; tune/validate before clinical interpretation."""
    prev_var = max(float(previous_variance), 1e-6)
    rel = float(np.clip(observation_reliability, 1e-3, 1.0))
    obs_var = 1.0 / rel
    gain = prev_var / (prev_var + obs_var)
    updated = float(previous_risk + gain * (new_risk - previous_risk))
    updated_var = float((1 - gain) * prev_var)
    return {"updated_risk": float(np.clip(updated, 0, 1)),
            "posterior_variance": updated_var, "update_gain": float(gain)}

def longitudinal_summary(history):
    """history requires assessment_date and a numeric movement-derived state or risk."""
    h = history.copy()
    h["assessment_date"] = pd.to_datetime(h["assessment_date"], errors="coerce")
    h = h.dropna(subset=["assessment_date", "estimated_risk"]).sort_values("assessment_date")
    if len(h) < 2:
        return {"n_observations": len(h), "change_point_flag": False,
                "trajectory_slope_per_day": None, "note": "At least two dated observations required."}
    days = (h["assessment_date"] - h["assessment_date"].iloc[0]).dt.total_seconds()/86400
    vals = pd.to_numeric(h["estimated_risk"], errors="coerce").to_numpy()
    slope = float(np.polyfit(days.to_numpy(), vals, 1)[0]) if np.ptp(days.to_numpy()) > 0 else 0.0
    delta = float(vals[-1] - vals[-2])
    # Transparent screening signal only; threshold must be pre-registered and validated.
    change_flag = bool(abs(delta) >= 0.15)
    return {"n_observations": int(len(h)), "change_point_flag": change_flag,
            "latest_change": delta, "trajectory_slope_per_day": slope,
            "note": "Exploratory change signal; not a clinical deterioration/diagnosis rule."}

def recommendation_stability(recommend_fn, base_args, perturbations, top_k=3):
    """Compares top-ranked option IDs under user-specified plausible perturbations."""
    rankings = []
    for args in [base_args] + perturbations:
        ranked, _ = recommend_fn(**args)
        rankings.append(list(ranked["option_id"].head(top_k)) if not ranked.empty else [])
    base = set(rankings[0])
    overlaps = [len(base.intersection(set(r))) / max(1, len(base)) for r in rankings[1:]]
    return {"top_k": top_k, "rankings": rankings,
            "mean_topk_overlap": float(np.mean(overlaps)) if overlaps else 1.0,
            "stable_at_80pct_overlap": bool(np.mean(overlaps) >= 0.8) if overlaps else True}

# Conformal monitoring placeholder: only valid after calibration on an independent, representative cohort.
# Do not report conformal coverage unless exchangeability/calibration assumptions are assessed.

## 13. Stage 8 — compact model, pruning proxy, quantization assessment, and latency profile

In [ ]:
import time, joblib
from sklearn.linear_model import LogisticRegression

# Feature dimensionality already reduced by low-rank bilinear interactions.
# Export the calibrated baseline and preprocessing objects for reproducibility.
joblib.dump({
    "calibrator": calibrator,
    "motion_imputer": motion_imputer, "motion_scaler": motion_scaler,
    "mri_imputer": mri_imputer, "mri_scaler": mri_scaler,
    "motion_columns": motion_cols, "mri_columns": mri_cols,
    "cca": cca, "random_state": RANDOM_STATE,
    "model_name": "NeuroRelAD-NMR-Fuse-baseline"
}, OUTPUT_DIR / "neurorelad_model.joblib")

def profile_batch_one(model, X, repeats=100):
    if len(X) == 0:
        return {"error": "No test rows"}
    sample = X[:1]
    for _ in range(10):
        model.predict_proba(sample)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        model.predict_proba(sample)
        times.append((time.perf_counter() - t0) * 1000)
    return {"batch_size": 1, "repeats": repeats,
            "median_latency_ms": float(np.median(times)),
            "p95_latency_ms": float(np.percentile(times, 95)),
            "mean_latency_ms": float(np.mean(times))}

latency = profile_batch_one(calibrator, F_te)
# Parameter count is not directly exposed for sklearn logistic regression; count stored coefficients.
coef_count = int(sum(np.size(a) for a in base.coef_.ravel())) if False else int(base.coef_.size + base.intercept_.size)
complexity = {
    "fused_feature_count": int(F_tr.shape[1]),
    "linear_model_coefficient_count": coef_count,
    "serialized_model_size_bytes": int((OUTPUT_DIR / "neurorelad_model.joblib").stat().st_size),
    **latency,
    "note": "CPU/software latency measured in this runtime; report hardware/software details with results."
}
(OUTPUT_DIR / "computational_profile.json").write_text(json.dumps(complexity, indent=2))
print(json.dumps(complexity, indent=2))

## 14. Experimental validation — subject-level CV, unimodal comparison, calibration, bootstrap CI

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
import matplotlib.pyplot as plt

def evaluate_group_cv(X, y, groups, n_splits=5):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for fold, (tr, te) in enumerate(splitter.split(X, y, groups), 1):
        if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
            rows.append({"fold": fold, "error": "Only one class in fold; reduce folds or collect more subjects"})
            continue
        model = make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True),
            StandardScaler(),
            LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1, solver="liblinear")
        )
        model.fit(X[tr], y[tr])
        p = model.predict_proba(X[te])[:, 1]
        yp = (p >= 0.5).astype(int)
        rows.append({"fold": fold, "n_test": len(te),
                     "n_test_subjects": len(set(np.asarray(groups)[te])),
                     "accuracy": accuracy_score(y[te], yp),
                     "f1": f1_score(y[te], yp, zero_division=0),
                     "roc_auc": roc_auc_score(y[te], p),
                     "brier": brier_score_loss(y[te], p)})
    return pd.DataFrame(rows)

# Nested subject-level CV should be used for final model selection. This grouped CV is a transparent baseline.
groups_all = paired["participant_id"].astype(str).to_numpy()
y_all = paired["cp_label"].astype(int).to_numpy()
X_motion = paired[motion_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
X_mri = paired[mri_cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)

cv_motion = evaluate_group_cv(X_motion, y_all, groups_all, n_splits=min(5, max(2, paired["participant_id"].nunique())))
cv_mri = evaluate_group_cv(X_mri, y_all, groups_all, n_splits=min(5, max(2, paired["participant_id"].nunique())))
cv_motion["modality"] = "movement_only"
cv_mri["modality"] = "MRI_only"
cv_unimodal = pd.concat([cv_motion, cv_mri], ignore_index=True)
cv_unimodal.to_csv(OUTPUT_DIR / "unimodal_group_cv.csv", index=False)
display(cv_unimodal)

In [ ]:
# Bootstrap confidence intervals over held-out participants (cluster bootstrap).
def cluster_bootstrap_ci(predictions, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    ids = predictions["participant_id"].astype(str).unique()
    stats = []
    for _ in range(n_boot):
        sampled_ids = rng.choice(ids, size=len(ids), replace=True)
        pieces = [predictions[predictions["participant_id"].astype(str) == sid] for sid in sampled_ids]
        b = pd.concat(pieces, ignore_index=True)
        if b["cp_label"].nunique() < 2:
            continue
        stats.append({
            "accuracy": accuracy_score(b["cp_label"], b["predicted_label"]),
            "f1": f1_score(b["cp_label"], b["predicted_label"], zero_division=0),
            "roc_auc": roc_auc_score(b["cp_label"], b["estimated_risk"]),
            "brier": brier_score_loss(b["cp_label"], b["estimated_risk"])
        })
    boot = pd.DataFrame(stats)
    if boot.empty:
        return pd.DataFrame()
    return pd.DataFrame({
        "metric": boot.columns,
        "lower_95": [boot[c].quantile(.025) for c in boot],
        "median": [boot[c].median() for c in boot],
        "upper_95": [boot[c].quantile(.975) for c in boot]
    })

pred_df = pd.read_csv(OUTPUT_DIR / "heldout_predictions.csv")
ci_df = cluster_bootstrap_ci(pred_df)
ci_df.to_csv(OUTPUT_DIR / "bootstrap_confidence_intervals.csv", index=False)
display(ci_df)

In [ ]:
# Calibration plot for held-out risk estimates
from sklearn.calibration import calibration_curve
if pred_df["cp_label"].nunique() == 2:
    frac_pos, mean_pred = calibration_curve(pred_df["cp_label"], pred_df["estimated_risk"], n_bins=8, strategy="quantile")
    plt.figure(figsize=(6.5, 5.5))
    plt.plot(mean_pred, frac_pos, marker="o", linewidth=2, label="NeuroRelAD NMR-Fuse")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ideal")
    plt.xlabel("Mean predicted risk")
    plt.ylabel("Observed CP-label fraction")
    plt.title("Held-out calibration")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "calibration_plot.png", dpi=300, bbox_inches="tight")
    plt.show()

## 15. Component ablation and multimodal comparison

In [ ]:
def grouped_cv_for_frame(frame, cols, label_col="cp_label", group_col="participant_id", n_splits=5):
    X = frame[cols].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
    y = frame[label_col].astype(int).to_numpy()
    groups = frame[group_col].astype(str).to_numpy()
    return evaluate_group_cv(X, y, groups, n_splits=min(n_splits, max(2, len(np.unique(groups)))))

# Feature-level ablations that can be tested without leaking held-out subjects.
ablation_rows = []
for label, cols in [
    ("movement_only", motion_cols),
    ("MRI_only", mri_cols),
    ("movement_plus_MRI_raw", motion_cols + mri_cols),
]:
    res = grouped_cv_for_frame(paired, cols)
    res["ablation"] = label
    ablation_rows.append(res)
ablation = pd.concat(ablation_rows, ignore_index=True)
ablation.to_csv(OUTPUT_DIR / "component_ablation_group_cv.csv", index=False)
display(ablation)

## 16. Results package and reproducibility manifest

In [ ]:
import sklearn, scipy
try:
    import nibabel
    nibabel_version = nibabel.__version__
except Exception:
    nibabel_version = None

run_manifest = {
    "model": "NeuroRelAD — Neuro-Motor Relational and Adaptive Development",
    "database": "IMB-MD",
    "random_state": RANDOM_STATE,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "nibabel": nibabel_version,
    "data_root": str(DATA_ROOT),
    "manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    "paired_verified_rows": int(df["paired_verified"].sum()),
    "paired_labeled_rows": int(len(paired)),
    "unique_paired_labeled_participants": int(paired["participant_id"].nunique()),
    "outputs": sorted([p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print(json.dumps(run_manifest, indent=2))
print("\nOutputs written to:", OUTPUT_DIR)

## 17. Before reporting results

1. Confirm consent, ethics approval, de-identification, data-use permissions, and MRI/pose linkage with the data custodian.
2. Audit participant overlap across folds and sites; use nested grouped CV for model selection and an independent cohort for final testing.
3. Document MRI sequence/field strength, voxel spacing, motion capture/pose-estimation system, missingness, age distribution, label definition, adjudication, and class balance.
4. Replace generic intensity summaries with validated anatomical-region features if claiming region-specific neuro-motor associations.
5. Populate the pathway knowledge base with clinician-reviewed, versioned guidance and verify each source's applicability.
6. Pre-register thresholds and uncertainty procedures; evaluate calibration, subgroup performance, external validity, and decision consequences.
7. Treat output recommendations as clinician-review candidates only. Do not claim causal effects, diagnostic readiness, or clinical benefit from these analyses alone.
8. Do not describe IMB-MD as publicly uploaded to Kaggle unless you actually complete the upload and have permission to distribute every included record.